# Search Strategy Refinement, Method Comparison

*Author: Regina Chua*

> This notebook tests three complementary methods for refining the search strategy and pre-ranking
> the corpus against a small set of manually confirmed seed papers. The goal is to compare which
> method surfaces the most relevant articles and what new vocabulary it suggests, feeding back into
> `search_strategy.py` before the full multi-database run.

**Methods tested:**

| Method | What it does | Best for |
|---|---|---|
| **TF-IDF** | Term weighting + cosine similarity to seed query | Baseline ranking; corpus-level keyword surfacing |
| **BM25** | Term-frequency ranking with length normalisation | Keyword recall, interpretable scores |
| **KeyBERT / YAKE** | Keyphrase extraction from seed abstracts | Surfacing new vocabulary for query expansion |
| **SPECTER embeddings** | Semantic similarity in dense vector space | Finding papers that use different terminology |

**Workflow:** populate `seed_papers.csv` → run top-to-bottom → inspect rankings and candidate
keyphrases → update `search_strategy.py` with any new terms.

---

### Seed set guidance

> For reliable ranking:
>
> | Size | What it enables |
> |---|---|
> | **≥ 15** | Minimum for stable BM25 and embedding centroids |
> | **20–30** | Recommended — covers enough sub-theme diversity |
> | **≥ 50** | Needed for the calibration step (precision/recall tuning) |

## 1. Environment Setup

In [ ]:
%pip install --upgrade pip
%pip install rank-bm25 sentence-transformers keybert yake --quiet

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 100)

SEED_PATH   = Path("seed_papers.csv")
# Clean UTF-8 export of the same seed papers — run build_refinement_corpus.py after seed updates.
CORPUS_PATH = Path("refinement_corpus.csv")
TOP_N       = 20   # how many top-ranked corpus papers to show per method


def _normalize_term(s):
    s = re.sub(r"[^a-z0-9\s]", " ", s.lower())
    return re.sub(r"\s+", " ", s).strip()


def _criteria_term_set(criteria_dict):
    terms = set()
    for group in criteria_dict.values():
        for t in group:
            terms.add(_normalize_term(t.replace("*", "")))
    return terms


def _strategy_term_set():
    """Normalized terms in the full live strategy (for BM25 / KeyBERT dedup)."""
    from search_strategy import ALTERNATE_TERMS, EXCLUSION_TERMS, INCLUSION_CRITERIA

    terms = _criteria_term_set(INCLUSION_CRITERIA) | _criteria_term_set(ALTERNATE_TERMS)
    for t in EXCLUSION_TERMS:
        terms.add(_normalize_term(t.replace("*", "")))
    return terms


def _initial_criteria_term_set():
    """Normalized terms in INITIAL_CRITERIA (for TF-IDF expansion baseline)."""
    from search_strategy import INITIAL_CRITERIA

    return _criteria_term_set(INITIAL_CRITERIA)


def _matches_initial(term_norm, initial_terms):
    if term_norm in initial_terms:
        return True
    return any(
        len(init) > 3 and (init in term_norm or term_norm in init)
        for init in initial_terms
    )


def _build_category_anchors():
    """Token anchors per category for assigning new terms to disease / spatial / exposure."""
    from search_strategy import ALTERNATE_TERMS, INCLUSION_CRITERIA, INITIAL_CRITERIA

    anchors = {cat: set() for cat in ("disease", "spatial", "exposure")}
    for criteria in (INITIAL_CRITERIA, INCLUSION_CRITERIA, ALTERNATE_TERMS):
        for cat, terms in criteria.items():
            for t in terms:
                norm = _normalize_term(t.replace("*", ""))
                anchors[cat].add(norm)
                anchors[cat].update(norm.split())
    return anchors


def _classify_term(term_norm, category_anchors):
    term_tokens = set(term_norm.split())
    best_cat, best_score = "exposure", -1
    for cat, anchor in category_anchors.items():
        score = len(term_tokens & anchor) + sum(
            1 for a in anchor if len(a) > 3 and a in term_norm
        )
        if score > best_score:
            best_cat, best_score = cat, score
    return best_cat


existing_strategy_terms = _strategy_term_set()
initial_criteria_terms = _initial_criteria_term_set()
category_anchors = _build_category_anchors()

print("Environment ready.")

## 2. Load Seed Papers & Corpus

> The seed set is loaded from `seed_papers.csv` (55 confirmed-relevant papers). The **corpus**
> is the same papers, cleaned and exported to `refinement_corpus.csv` by
> `build_refinement_corpus.py` (encoding fixes, title + abstract required). Re-run that
> script whenever `seed_papers.csv` changes.

In [ ]:
def _combine_text(row):
    """Join title, abstract, and keywords into one string for indexing."""
    parts = [
        str(row.get("title", "") or ""),
        str(row.get("abstract", "") or ""),
        str(row.get("keywords", "") or ""),
    ]
    return " ".join(p for p in parts if p).strip()


def _tokenize(text):
    """Lowercase, strip punctuation, split on whitespace."""
    return re.sub(r"[^a-z0-9\s]", " ", text.lower()).split()


# --- Seed papers ---
if not SEED_PATH.exists():
    raise FileNotFoundError(
        f"'{SEED_PATH}' not found. Add your confirmed-relevant papers to that file "
        "(title + abstract required) and re-run this cell."
    )

df_seed = pd.read_csv(SEED_PATH, encoding="latin-1")
# Drop the placeholder row if it hasn't been replaced yet
df_seed = df_seed[~df_seed["title"].astype(str).str.startswith("REPLACE")].reset_index(drop=True)

if len(df_seed) == 0:
    raise ValueError(
        "seed_papers.csv contains no real entries yet. Fill in at least one confirmed-relevant "
        "paper (title + abstract) and re-run."
    )

df_seed["_text"] = df_seed.apply(_combine_text, axis=1)
seed_texts = df_seed["_text"].tolist()
print(f"Seed papers loaded: {len(df_seed)}")
if len(df_seed) < 15:
    print(f"  ⚠  {len(df_seed)} papers is below the recommended minimum of 15. "
          "Rankings will be less reliable — add more confirmed-relevant papers.")
display(df_seed[[c for c in ["title","doi","pubmed_id"] if c in df_seed.columns]].head())

# --- Corpus ---
if not CORPUS_PATH.exists():
    raise FileNotFoundError(
        f"'{CORPUS_PATH}' not found. Run build_refinement_corpus.py first:\n"
        "  python build_refinement_corpus.py"
    )

df_corpus = pd.read_csv(CORPUS_PATH, encoding="utf-8")
df_corpus["_text"] = df_corpus.apply(_combine_text, axis=1)
# Drop rows with no usable text
df_corpus = df_corpus[df_corpus["_text"].str.strip().astype(bool)].reset_index(drop=True)
print(f"\nCorpus loaded: {len(df_corpus)} documents from '{CORPUS_PATH.name}'")

## 3. Method 1 — TF-IDF

> Baseline method (same family as `pubmed.ipynb`). **Article ranking:** cosine similarity
> between each corpus document and the concatenated seed text in a shared TF-IDF space.
> **Keywords:** corpus-level 1–3 gram terms ranked by TF-IDF + frequency, compared against
> `INITIAL_CRITERIA` in `search_strategy.py`. New terms are assigned to disease / spatial /
> exposure and saved in `df_tfidf_expanded` for comparison with later methods.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

NGRAM_RANGE = (1, 3)
MAX_DF = 0.95

# --- Articles: query–document cosine similarity ---
doc_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), max_df=MAX_DF)
X_corpus = doc_vectorizer.fit_transform(df_corpus["_text"])
X_query = doc_vectorizer.transform([" ".join(seed_texts)])

df_tfidf = df_corpus.copy()
df_tfidf["tfidf_score"] = cosine_similarity(X_query, X_corpus)[0]
df_tfidf = df_tfidf.sort_values("tfidf_score", ascending=False).reset_index(drop=True)

print(f"Top {TOP_N} TF-IDF-ranked articles (cosine similarity to seed query):")
show_cols = [c for c in ["title", "tfidf_score", "doi", "publication_date"] if c in df_tfidf.columns]
display(df_tfidf[show_cols].head(TOP_N).style.format({"tfidf_score": "{:.3f}"}))

# --- Keywords: corpus-level n-gram ranking ---
kw_vectorizer = TfidfVectorizer(ngram_range=NGRAM_RANGE, stop_words="english", max_df=MAX_DF)
X_kw = kw_vectorizer.fit_transform(df_corpus["_text"])
kw_tfidf = np.asarray(X_kw.sum(axis=0)).ravel()
kw_names = kw_vectorizer.get_feature_names_out()

count_vec = CountVectorizer(ngram_range=NGRAM_RANGE, stop_words="english")
Y = count_vec.fit_transform(df_corpus["_text"])
kw_counts = np.asarray(Y.sum(axis=0)).ravel()

tfidf_keywords = (
    pd.DataFrame({"term": kw_names, "tfidf": kw_tfidf})
    .merge(
        pd.DataFrame({"term": count_vec.get_feature_names_out(), "count": kw_counts}),
        on="term",
        how="outer",
    )
    .fillna(0)
)
tfidf_keywords["ngram_len"] = tfidf_keywords["term"].str.count(" ") + 1
tfidf_keywords["score"] = (
    tfidf_keywords["tfidf"] * 0.7
    + (tfidf_keywords["count"] / (tfidf_keywords["count"].max() + 1e-9)) * 0.3
)
tfidf_keywords["term_norm"] = tfidf_keywords["term"].apply(_normalize_term)
tfidf_keywords["already_in_initial"] = tfidf_keywords["term_norm"].apply(
    lambda t: _matches_initial(t, initial_criteria_terms)
)
tfidf_keywords = tfidf_keywords[
    (tfidf_keywords["term"].str.len() > 2) & (~tfidf_keywords["term"].str.match(r"^\d+$"))
]
tfidf_keywords = tfidf_keywords.sort_values("score", ascending=False).reset_index(drop=True)

print(f"\nTop {TOP_N} corpus keywords (TF-IDF + frequency):")
display(
    tfidf_keywords[["term", "ngram_len", "count", "tfidf", "score", "already_in_initial"]]
    .head(TOP_N)
    .style.format({"tfidf": "{:.2f}", "score": "{:.2f}"})
)

# Expanded terms by category (new vs INITIAL_CRITERIA) — for cross-method comparison
new_tfidf = tfidf_keywords[~tfidf_keywords["already_in_initial"]].copy()
new_tfidf["category"] = new_tfidf["term_norm"].apply(
    lambda t: _classify_term(t, category_anchors)
)

expansions_by_cat = {
    cat: (
        new_tfidf[new_tfidf["category"] == cat]
        .sort_values("score", ascending=False)["term"]
        .head(TOP_N)
        .tolist()
    )
    for cat in ("disease", "spatial", "exposure")
}
max_rows = max((len(v) for v in expansions_by_cat.values()), default=1)

df_tfidf_expanded = pd.DataFrame({
    cat: expansions_by_cat[cat] + [""] * (max_rows - len(expansions_by_cat[cat]))
    for cat in ("disease", "spatial", "exposure")
})

df_tfidf_expansions = (
    new_tfidf.sort_values("score", ascending=False)
    .groupby("category", sort=False)
    .head(TOP_N)
    .reset_index(drop=True)[["category", "term", "score", "tfidf", "count"]]
)
df_tfidf_expansions.insert(0, "method", "tfidf")

print(f"\nTF-IDF expanded terms by category (top {TOP_N} new vs INITIAL_CRITERIA):")
display(df_tfidf_expanded.replace("", pd.NA))

In [ ]:
from search_strategy import INITIAL_CRITERIA


def _print_list(title, items):
    print(title)
    if not items:
        print("  (none)")
    else:
        for item in items:
            print(f"  - {item}")
    print()


# 1. All terms in INITIAL_CRITERIA
initial_terms = sorted(
    {t for group in INITIAL_CRITERIA.values() for t in group},
    key=str.lower,
)

# 2. TF-IDF keywords that overlap INITIAL_CRITERIA
tfidf_matched = tfidf_keywords[tfidf_keywords["already_in_initial"]]["term"].tolist()

# 3. Top 20 TF-IDF keywords not in INITIAL_CRITERIA
tfidf_new = (
    tfidf_keywords[~tfidf_keywords["already_in_initial"]]
    .head(TOP_N)["term"]
    .tolist()
)

_print_list(f"1. In INITIAL_CRITERIA ({len(initial_terms)} terms):", initial_terms)
_print_list(f"2. TF-IDF matched to INITIAL_CRITERIA ({len(tfidf_matched)} terms):", tfidf_matched)
_print_list(f"3. TF-IDF new — top {TOP_N} not in INITIAL_CRITERIA:", tfidf_new)

## 4. Method 2 — BM25

> BM25 extends TF-IDF with document-length normalisation and term saturation. **Article
> ranking:** score each corpus document against the concatenated seed query. **Keywords:**
> seed-query tokens weighted by BM25 IDF (rare in the corpus × frequent in the query).
> Compare rankings and keyword lists with Section 3 to see what BM25 adds.

In [ ]:
from collections import Counter

from rank_bm25 import BM25Okapi

# --- Articles ---
corpus_tokens = [_tokenize(t) for t in df_corpus["_text"]]
bm25 = BM25Okapi(corpus_tokens)

seed_query_tokens = _tokenize(" ".join(seed_texts))
bm25_scores = bm25.get_scores(seed_query_tokens)

df_bm25 = df_corpus.copy()
df_bm25["bm25_score"] = bm25_scores
df_bm25 = df_bm25.sort_values("bm25_score", ascending=False).reset_index(drop=True)

print(f"Top {TOP_N} BM25-ranked articles:")
show_cols = [c for c in ["title", "bm25_score", "doi", "publication_date"] if c in df_bm25.columns]
display(df_bm25[show_cols].head(TOP_N).style.format({"bm25_score": "{:.2f}"}))

# --- Keywords: query tokens weighted by BM25 IDF ---
query_counts = Counter(seed_query_tokens)
bm25_keywords = (
    pd.DataFrame([
        {"term": term, "query_freq": query_counts[term], "idf": bm25.idf.get(term, 0.0)}
        for term in query_counts
        if term in bm25.idf and len(term) > 2
    ])
    .assign(bm25_weight=lambda d: d["query_freq"] * d["idf"])
    .sort_values("bm25_weight", ascending=False)
    .reset_index(drop=True)
)
bm25_keywords["term_norm"] = bm25_keywords["term"].apply(_normalize_term)
bm25_keywords["already_in_strategy"] = bm25_keywords["term_norm"].isin(existing_strategy_terms)

print(f"\nTop {TOP_N} query keywords (BM25 IDF x seed query frequency):")
display(
    bm25_keywords[["term", "query_freq", "idf", "bm25_weight", "already_in_strategy"]]
    .head(TOP_N)
    .style.format({"idf": "{:.2f}", "bm25_weight": "{:.2f}"})
)

## 5. Method 3 — Keyphrase Extraction (KeyBERT + YAKE)

> Both methods extract keyphrases from the seed abstracts, but use different signals. **YAKE** uses
> statistical co-occurrence (fast, no model download). **KeyBERT** uses contextual embeddings (richer,
> slower). The union of their output is a candidate list of terms to add to `search_strategy.py`
> — specifically to `ALTERNATE_TERMS` or `INCLUSION_CRITERIA`. I mark any term that already exists
> in the strategy so it's easy to spot the genuinely new ones.

In [ ]:
import yake
from keybert import KeyBERT

from search_strategy import INCLUSION_CRITERIA, ALTERNATE_TERMS

# All terms already in the strategy (for deduplication display)
existing_terms = set()
for group in list(INCLUSION_CRITERIA.values()) + list(ALTERNATE_TERMS.values()):
    for t in group:
        existing_terms.add(t.lower().replace("*", ""))

seed_corpus_text = " ".join(seed_texts)

# --- YAKE ---
yake_extractor = yake.KeywordExtractor(
    lan="en", n=3, dedupLim=0.7, top=30, features=None
)
yake_kws = yake_extractor.extract_keywords(seed_corpus_text)
# YAKE scores are inverted (lower = more important)
yake_df = pd.DataFrame(yake_kws, columns=["keyphrase", "yake_score"]).sort_values("yake_score")
yake_df["already_in_strategy"] = yake_df["keyphrase"].str.lower().isin(existing_terms)

print("--- YAKE keyphrases (lower score = more relevant) ---")
display(yake_df.head(20).style.format({"yake_score": "{:.4f}"}))

# --- KeyBERT ---
# Uses a lightweight all-MiniLM model by default (fast). Swap for 'allenai-specter'
# if you want scientific-domain embeddings (requires the SPECTER model to be downloaded first).
print("\nLoading KeyBERT model (this may take a moment on first run)...")
kw_model = KeyBERT()
keybert_kws = kw_model.extract_keywords(
    seed_corpus_text,
    keyphrase_ngram_range=(1, 3),
    stop_words="english",
    top_n=30,
    diversity=0.6,   # MMR diversity — avoids near-duplicate keyphrases
)
keybert_df = pd.DataFrame(keybert_kws, columns=["keyphrase", "keybert_score"]).sort_values(
    "keybert_score", ascending=False
)
keybert_df["already_in_strategy"] = keybert_df["keyphrase"].str.lower().isin(existing_terms)

print("\n--- KeyBERT keyphrases (higher score = more relevant) ---")
display(keybert_df.head(20).style.format({"keybert_score": "{:.3f}"}))

# --- Union of new terms ---
new_yake = set(yake_df[~yake_df["already_in_strategy"]]["keyphrase"].str.lower())
new_keybert = set(keybert_df[~keybert_df["already_in_strategy"]]["keyphrase"].str.lower())
new_terms_union = sorted(new_yake | new_keybert)
print(f"\n{len(new_terms_union)} candidate new terms (not already in search_strategy.py):")
for t in new_terms_union:
    print(f"  {t}")

## 6. Method 4 — SPECTER Semantic Similarity

> SPECTER is a transformer model trained specifically for scientific document similarity. I embed
> each seed paper and each corpus document, then score each corpus document by its **cosine
> similarity to the seed centroid** (the average of the seed embeddings). This catches papers that
> are conceptually similar to the seed set but use different terminology — the gap that keyword
> methods leave open.
>
> First run downloads the SPECTER model (~400 MB); subsequent runs use the cached version.
> If the download is too slow, replace `'allenai-specter'` with `'all-MiniLM-L6-v2'` for a
> faster ~80 MB model (slightly less domain-specific).

In [ ]:
from sentence_transformers import SentenceTransformer, util

MODEL_NAME = "allenai-specter"   # swap for "all-MiniLM-L6-v2" if you want a faster, smaller model

print(f"Loading {MODEL_NAME} (downloads ~400 MB on first run, then cached) ...")
model = SentenceTransformer(MODEL_NAME)
print("Model loaded.")

# Embed seed papers
print("Embedding seed papers ...")
seed_embeddings = model.encode(seed_texts, show_progress_bar=True, convert_to_tensor=True)
seed_centroid = seed_embeddings.mean(dim=0)  # single representative vector

# Embed corpus
print("Embedding corpus (may take a few minutes) ...")
corpus_embeddings = model.encode(
    df_corpus["_text"].tolist(), show_progress_bar=True, convert_to_tensor=True
)

# Score each corpus document against the seed centroid
sim_scores = util.cos_sim(seed_centroid.unsqueeze(0), corpus_embeddings)[0].cpu().numpy()

df_specter = df_corpus.copy()
df_specter["specter_sim"] = sim_scores
df_specter = df_specter.sort_values("specter_sim", ascending=False).reset_index(drop=True)

print(f"\nTop {TOP_N} SPECTER-ranked corpus documents:")
show_cols = [c for c in ["title", "specter_sim", "doi", "publication_date"] if c in df_specter.columns]
display(df_specter[show_cols].head(TOP_N).style.format({"specter_sim": "{:.3f}"}))

## 7. Compare Rankings

> Merges the BM25 and SPECTER rankings into one table so it's easy to spot:
> - Papers that rank high on **both** — very likely relevant.
> - Papers high on SPECTER but low on BM25 — semantically similar but may use different vocabulary;
>   their titles/abstracts are good candidates for new search terms.
> - Papers high on BM25 but low on SPECTER — keyword-rich but may be thematically tangential;
>   worth a quick scan to check whether they are noise or whether the seed set is missing a sub-theme.

In [ ]:
# Normalise both scores to [0, 1] for a fair side-by-side comparison
def _norm(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn) if mx > mn else s * 0


bm25_rank = df_bm25[["title", "doi", "bm25_score"]].copy()
bm25_rank["bm25_norm"] = _norm(bm25_rank["bm25_score"])
bm25_rank["bm25_rank"] = bm25_rank["bm25_score"].rank(ascending=False).astype(int)

specter_rank = df_specter[["title", "doi", "specter_sim"]].copy()
specter_rank["specter_norm"] = _norm(specter_rank["specter_sim"])
specter_rank["specter_rank"] = specter_rank["specter_sim"].rank(ascending=False).astype(int)

merged = bm25_rank.merge(specter_rank[["doi", "specter_norm", "specter_rank"]], on="doi", how="inner")
merged["combined_norm"] = (merged["bm25_norm"] + merged["specter_norm"]) / 2
merged = merged.sort_values("combined_norm", ascending=False).reset_index(drop=True)

print(f"Top {TOP_N} papers by combined BM25 + SPECTER score:")
display(
    merged[["title", "bm25_rank", "specter_rank", "bm25_norm", "specter_norm", "combined_norm"]]
    .head(TOP_N)
    .style.format({
        "bm25_norm":     "{:.3f}",
        "specter_norm":  "{:.3f}",
        "combined_norm": "{:.3f}",
    })
    .background_gradient(subset=["combined_norm"], cmap="YlGn")
)

# Flag interesting divergences
divergent = merged[abs(merged["bm25_rank"] - merged["specter_rank"]) > 50].head(10)
if not divergent.empty:
    print(f"\nPapers with large rank divergence (>50 positions) — worth investigating:")
    display(divergent[["title", "bm25_rank", "specter_rank"]].reset_index(drop=True))

## 8. Export Ranked Results

> Save all three score columns to a CSV so the rankings can be reviewed outside the notebook and
> used as a reference when updating `search_strategy.py`.

In [ ]:
from pathlib import Path

# Attach scores back to the full corpus metadata
full_ranked = df_corpus.drop(columns=["_text"]).merge(
    merged[["doi", "bm25_rank", "specter_rank", "bm25_norm", "specter_norm", "combined_norm"]],
    on="doi",
    how="left",
)
full_ranked = full_ranked.sort_values("combined_norm", ascending=False).reset_index(drop=True)

output_path = Path("query_refinement_ranked.csv")
full_ranked.to_csv(output_path, index=False)
print(f"Exported {len(full_ranked)} ranked records to {output_path.resolve()}")
print()
print("Next steps:")
print("  1. Review new keywords from Sections 3–5 (TF-IDF, BM25, KeyBERT/YAKE) and add useful")
print("     ones to ALTERNATE_TERMS (or INCLUSION_CRITERIA) in search_strategy.py.")
print("  2. Skim the top-ranked papers from Section 7 — check whether any sub-themes are")
print("     under-represented in the seed set.")
print("  3. If enough papers diverge between BM25 and SPECTER, consider adding a sub-theme")
print("     to the seed set to cover that vocabulary gap.")
print("  4. Once you have ≥50 labelled papers (include + confirmed exclude), run the")
print("     calibration step: score a held-out set and measure precision/recall to tune")
print("     the similarity thresholds before running LLM screening in prescreen.ipynb.")